# 06c — Paired Bootstrap Significance Tests (Final)

**Why 06b was still wrong:**  
`ensemble_probs()` averaged probabilities across seeds — but each seed produces its
**own test partition** (different samples, same size). Averaging probs for different
samples is mathematically invalid and produces F1≈0.73 (random noise).
The per-seed validation passed because each seed's probs ARE aligned with that seed's
labels individually; the cross-seed mean is what breaks.

**Fix — single-seed bootstrap on seed=42:**  
Standard practice in the ML literature. The bootstrap operates on one fixed
test partition (seed=42), so probs and labels are guaranteed to align.
Seeds 1337 and 2024 are checked for directional consistency (signs of Δ),
and their individual p-values are reported alongside seed=42 as supplementary.

**Paper reporting:**  
> "Statistical significance was assessed via paired bootstrap (B=10,000,
>  one-tailed, Efron–Tibshirani shift) on the seed=42 test partition.
>  Results for seeds 1337 and 2024 showed consistent sign of Δ across
>  all significant comparisons."

**Required datasets:**  
`06-fusion-v3-dataset` · `03-splits-v3-dataset` · `03-triage-v3-dataset`
`lotl-nlp-outputs` · `lotl-gat-v3-outputs`


## Cell 1 — Locate all datasets

In [1]:
import os, sys, json, time, warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID : {RUN_ID}')
print(f'Python : {sys.version.split()[0]}')

def _rglob_first(root, pattern):
    root = Path(root)
    if not root.exists(): return None
    for p in root.rglob(pattern): return p
    return None

def _find_dir(root, marker):
    p = _rglob_first(root, marker)
    return p.parent if p else None

KI = Path('/kaggle/input/datasets/onkarkmane1501')

SPLITS_DIR = _find_dir(KI, 'train_seed42.parquet')

# Use UNIQUE markers that cannot collide across datasets
TRIAGE_PROBS = _find_dir(KI, 'xgb_full_seed42.json')  # only in triage dataset
if TRIAGE_PROBS: TRIAGE_PROBS = TRIAGE_PROBS.parent / 'probs'
if TRIAGE_PROBS and not TRIAGE_PROBS.exists():
    # Fallback: triage_results.json that lives UNDER a 'triage' folder
    for m in KI.rglob('triage_results.json'):
        cand = m.parent / 'probs'
        if cand.exists():
            TRIAGE_PROBS = cand
            break

NLP_PROBS  = _find_dir(KI, 'nlp_results.json')
if NLP_PROBS: NLP_PROBS = NLP_PROBS / 'probs'

GAT_PROBS  = None
for cand in KI.rglob('gat_v3'):
    GAT_PROBS = cand / 'probs'; break
if not GAT_PROBS:
    d = _find_dir(KI, 'gat_results.json')
    if d: GAT_PROBS = d / 'probs'

# Fusion probs: unique marker (stacking_test_probs not in any branch dataset)
FUSION_PROBS = _find_dir(KI, 'stacking_test_probs_seed42.npy')

required = {
    'Splits':        SPLITS_DIR,
    'Triage probs':  TRIAGE_PROBS,
    'NLP probs':     NLP_PROBS,
    'GAT probs':     GAT_PROBS,
    'Fusion probs':  FUSION_PROBS,
}
print()
all_ok = True
for name, path in required.items():
    ok = path is not None and Path(path).exists()
    print(f'  {"[OK]" if ok else "[MISSING]":8s} {name}: {path}')
    if not ok: all_ok = False

if not all_ok:
    raise RuntimeError('Missing datasets — attach them before running.')

# Verify no path collisions
paths = [p for p in [TRIAGE_PROBS, NLP_PROBS, GAT_PROBS, FUSION_PROBS] if p]
assert len(paths) == len(set(str(p) for p in paths)), \
    f'PATH COLLISION: two datasets resolved to the same directory!\n{paths}'
print('  [OK]     No path collisions between datasets.')

OUT_DIR = Path('/kaggle/working/bootstrap')
OUT_DIR.mkdir(parents=True, exist_ok=True)
BOOTSTRAP_PATH = OUT_DIR / 'bootstrap_results.json'
print(f'\nOutput: {BOOTSTRAP_PATH}')


Run ID : 20260510_030012
Python : 3.12.12

  [OK]     Splits: /kaggle/input/datasets/onkarkmane1501/03-splits-v3-dataset/splits
  [OK]     Triage probs: /kaggle/input/datasets/onkarkmane1501/03-triage-v3-dataset/triage/probs
  [OK]     NLP probs: /kaggle/input/datasets/onkarkmane1501/lotl-nlp-outputs/nlp/probs
  [OK]     GAT probs: /kaggle/input/datasets/onkarkmane1501/lotl-gat-v3-outputs/gat_v3/probs
  [OK]     Fusion probs: /kaggle/input/datasets/onkarkmane1501/06-fusion-v3-dataset/fusion/probs
  [OK]     No path collisions between datasets.

Output: /kaggle/working/bootstrap/bootstrap_results.json


## Cell 2 — Config + imports

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score, roc_curve
)

STRATEGIES = ['averaging', 'stacking', 'attention', 'gating', 'moe']
STRAT_LABELS = {
    'averaging': 'Averaging (geom. mean)',
    'stacking':  'Stacking (LR on probs)',
    'attention': 'Cross-modal attention',
    'gating':    'Gating (softmax weights)',
    'moe':       'Sparse MoE (top-1)',
}
SEEDS      = [42, 1337, 2024]
LABEL_COL  = 'label'
B_RESAMPLES = 10_000
ALPHA       = 0.05
RNG_SEED    = 0

# Bootstrap seed: single canonical partition
BOOTSTRAP_SEED = 42

# Best strategies from NB06/NB07 (val-tuned, 3-seed mean):
#   C2 F1, C2 AUROC  → Gating   (0.9089, 0.9967)
#   TPR@1%FPR        → Attention (0.9645)
PRIMARY_REF   = 'gating'
SECONDARY_REF = 'attention'

# Known NB06 metrics at threshold=0.5 for seed=42 only (validation anchor)
KNOWN_42 = {
    ('averaging', 'test'):    {'f1': 0.9707, 'auroc': 0.9956},
    ('averaging', 'test_c2'): {'f1': 0.9373, 'auroc': 0.9969},
    ('attention', 'test'):    {'f1': 0.9736, 'auroc': 0.9893},
    ('attention', 'test_c2'): {'f1': 0.9211, 'auroc': 0.9892},
    ('gating',    'test'):    {'f1': 0.9724, 'auroc': 0.9962},
    ('gating',    'test_c2'): {'f1': 0.9175, 'auroc': 0.9977},
    ('stacking',  'test'):    {'f1': 0.9499, 'auroc': 0.9875},
    ('stacking',  'test_c2'): {'f1': 0.7320, 'auroc': 0.9874},
    ('moe',       'test'):    {'f1': 0.9706, 'auroc': 0.9928},
    ('moe',       'test_c2'): {'f1': 0.9045, 'auroc': 0.9965},
}

print('Config loaded.')
print(f'  Bootstrap seed  : {BOOTSTRAP_SEED} (single canonical partition)')
print(f'  B_RESAMPLES     : {B_RESAMPLES:,}')
print(f'  Primary ref     : {PRIMARY_REF}')
print(f'  Secondary ref   : {SECONDARY_REF}')


Config loaded.
  Bootstrap seed  : 42 (single canonical partition)
  B_RESAMPLES     : 10,000
  Primary ref     : gating
  Secondary ref   : attention


## Cell 3 — Load seed=42 probs and validate

All probs are loaded for **seed=42 only** (the bootstrap partition).
Each fusion strategy's prob array is validated against the known NB06 F1@0.5
metric (tolerance 0.002). This catches path collisions and wrong files
before running the 30-min bootstrap.

Branch probs (XGBoost, CodeBERT, GATv2) are loaded for seed=42 for the
fusion-vs-branch comparison section.


In [3]:
def _load(path):
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f'Missing: {p}')
    arr = np.load(p).astype(np.float32)
    assert arr.ndim == 1, f'Expected 1D array, got shape {arr.shape}'
    assert ((arr >= 0) & (arr <= 1)).all(), f'Probs out of [0,1]: {p}'
    return arr

def load_labels(split, seed=BOOTSTRAP_SEED):
    parq = SPLITS_DIR / f'{split}_seed{seed}.parquet'
    return pd.read_parquet(parq, columns=[LABEL_COL])[LABEL_COL].astype(np.int32).values

def compute_f1_auroc(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    f1     = round(float(f1_score(y_true, y_pred, zero_division=0)), 4)
    auroc  = round(float(roc_auc_score(y_true, y_prob)), 4)
    return f1, auroc

# Load seed=42 fusion probs
PROBS = {}   # PROBS[strategy][split]
LABELS = {}  # LABELS[split]

print('Loading and validating seed=42 fusion probs...')
for split in ('test', 'test_c2'):
    LABELS[split] = load_labels(split, seed=42)
    for strat in STRATEGIES:
        PROBS.setdefault(strat, {})
        arr = _load(FUSION_PROBS / f'{strat}_{split}_probs_seed42.npy')
        assert len(arr) == len(LABELS[split]), (
            f'Shape mismatch {strat} {split}: probs={len(arr)} labels={len(LABELS[split])}')
        f1, auroc = compute_f1_auroc(LABELS[split], arr)
        known = KNOWN_42.get((strat, split), {})
        ok_f1    = abs(f1    - known.get('f1',    f1))    < 0.002
        ok_auroc = abs(auroc - known.get('auroc', auroc)) < 0.002
        status = 'OK  ' if (ok_f1 and ok_auroc) else 'FAIL'
        if not (ok_f1 and ok_auroc):
            raise ValueError(
                f'VALIDATION FAIL: {strat} {split} '
                f'f1={f1} (exp {known.get("f1","?")})  '
                f'auroc={auroc} (exp {known.get("auroc","?")})')
        print(f'  {status} [{strat:<10} {split:7s}] f1={f1:.4f} auroc={auroc:.4f}')
        PROBS[strat][split] = arr

# Load seed=42 branch probs
print('\nLoading branch probs (seed=42)...')
BRANCH = {}
for branch, probs_dir in [('XGBoost', TRIAGE_PROBS),
                           ('CodeBERT', NLP_PROBS),
                           ('GATv2',    GAT_PROBS)]:
    BRANCH[branch] = {}
    for split in ('test', 'test_c2'):
        arr = _load(probs_dir / f'{split}_probs_seed42.npy')
        f1, auroc = compute_f1_auroc(LABELS[split], arr)
        print(f'  OK   [{branch:<10} {split:7s}] f1={f1:.4f} auroc={auroc:.4f}')
        BRANCH[branch][split] = arr

# Confirm no two datasets returned identical arrays (catches path collisions)
print('\nChecking for path collisions...')
for split in ('test', 'test_c2'):
    arrays = {
        'XGBoost':  BRANCH['XGBoost'][split],
        'CodeBERT': BRANCH['CodeBERT'][split],
        'GATv2':    BRANCH['GATv2'][split],
        'gating':   PROBS['gating'][split],
        'attention':PROBS['attention'][split],
        'averaging':PROBS['averaging'][split],
    }
    names = list(arrays.keys())
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            if np.array_equal(arrays[names[i]], arrays[names[j]]):
                raise ValueError(
                    f'COLLISION: {names[i]} and {names[j]} on {split} are identical arrays!\n'
                    f'Check that TRIAGE_PROBS, NLP_PROBS, GAT_PROBS, FUSION_PROBS are distinct.')
print('  No collisions found.')
print('\nAll validations passed. Probs and labels are correctly aligned for seed=42.')


Loading and validating seed=42 fusion probs...
  OK   [averaging  test   ] f1=0.9707 auroc=0.9956
  OK   [stacking   test   ] f1=0.9499 auroc=0.9875
  OK   [attention  test   ] f1=0.9736 auroc=0.9893
  OK   [gating     test   ] f1=0.9724 auroc=0.9962
  OK   [moe        test   ] f1=0.9706 auroc=0.9928
  OK   [averaging  test_c2] f1=0.9373 auroc=0.9969
  OK   [stacking   test_c2] f1=0.7320 auroc=0.9874
  OK   [attention  test_c2] f1=0.9211 auroc=0.9892
  OK   [gating     test_c2] f1=0.9175 auroc=0.9977
  OK   [moe        test_c2] f1=0.9045 auroc=0.9965

Loading branch probs (seed=42)...
  OK   [XGBoost    test   ] f1=0.9724 auroc=0.9962
  OK   [XGBoost    test_c2] f1=0.9175 auroc=0.9977
  OK   [CodeBERT   test   ] f1=0.7671 auroc=0.9819
  OK   [CodeBERT   test_c2] f1=0.2735 auroc=0.9924
  OK   [GATv2      test   ] f1=0.9238 auroc=0.9745
  OK   [GATv2      test_c2] f1=0.6983 auroc=0.9802

Checking for path collisions...
  No collisions found.

All validations passed. Probs and labels are 

## Cell 4 — Per-seed directional consistency check

Before running the full bootstrap on seed=42, verify that seeds 1337 and 2024
show the **same sign of Δ** for the primary comparisons. This supports the
paper claim that significance is not seed-specific.

This cell does NOT compute p-values (that would require per-seed bootstrap,
3× the runtime). It simply checks that metric(ref) > metric(other) holds
across all three seeds for the key comparisons.


In [4]:
def _tpr_at_fpr(y_true, y_prob, fpr_thresh=0.01):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    return float(np.interp(fpr_thresh, fpr, tpr))

def all_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    return dict(
        f1          = round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        auroc       = round(float(roc_auc_score(y_true, y_prob)), 4),
        pr_auc      = round(float(average_precision_score(y_true, y_prob)), 4),
        tpr_at_1fpr = round(_tpr_at_fpr(y_true, y_prob), 4),
    )

print('=== Directional consistency across all 3 seeds ===')
print('(Checks metric(ref) > metric(other) sign — not p-values)\n')

for split, slabel in [('test', 'Balanced'), ('test_c2', 'C2')]:
    print(f'--- {slabel} test ---')
    print(f'  {"Comparison":<38}  {"F1":>6}  {"AUROC":>6}  {"PR-AUC":>7}  {"TPR@1%":>7}  Consistent?')
    print('  ' + '-'*80)

    for ref_strat in [PRIMARY_REF, SECONDARY_REF]:
        for other in [s for s in STRATEGIES if s != ref_strat]:
            signs = []
            for seed in SEEDS:
                y_true = load_labels(split, seed)
                p_ref  = _load(FUSION_PROBS / f'{ref_strat}_{split}_probs_seed{seed}.npy')
                p_oth  = _load(FUSION_PROBS / f'{other}_{split}_probs_seed{seed}.npy')
                m_ref  = all_metrics(y_true, p_ref)
                m_oth  = all_metrics(y_true, p_oth)
                seed_signs = {k: (m_ref[k] > m_oth[k]) for k in m_ref}
                signs.append(seed_signs)

            # All-seed agreement per metric
            consistent = all(
                signs[0][k] == signs[1][k] == signs[2][k]
                for k in ['f1', 'auroc', 'pr_auc', 'tpr_at_1fpr']
            )
            label = f'{ref_strat} > {other}'
            # Show seed=42 deltas
            y42   = load_labels(split, 42)
            p_r42 = _load(FUSION_PROBS / f'{ref_strat}_{split}_probs_seed42.npy')
            p_o42 = _load(FUSION_PROBS / f'{other}_{split}_probs_seed42.npy')
            m_r42 = all_metrics(y42, p_r42)
            m_o42 = all_metrics(y42, p_o42)
            flag = '\u2705' if consistent else '\u26a0\ufe0f'
            print(f'  {label:<38}'
                  f'  {m_r42["f1"]-m_o42["f1"]:>+6.4f}'
                  f'  {m_r42["auroc"]-m_o42["auroc"]:>+6.4f}'
                  f'  {m_r42["pr_auc"]-m_o42["pr_auc"]:>+7.4f}'
                  f'  {m_r42["tpr_at_1fpr"]-m_o42["tpr_at_1fpr"]:>+7.4f}'
                  f'  {flag}')
    print()

print('✅ = sign is consistent across all 3 seeds for all 4 metrics.')
print('⚠️  = at least one metric has inconsistent sign across seeds (bootstrap may yield ns).')


=== Directional consistency across all 3 seeds ===
(Checks metric(ref) > metric(other) sign — not p-values)

--- Balanced test ---
  Comparison                                  F1   AUROC   PR-AUC   TPR@1%  Consistent?
  --------------------------------------------------------------------------------
  gating > averaging                      +0.0017  +0.0006  +0.0005  +0.0015  ⚠️
  gating > stacking                       +0.0225  +0.0087  +0.0143  +0.0184  ✅
  gating > attention                      -0.0012  +0.0069  +0.0095  -0.0116  ⚠️
  gating > moe                            +0.0018  +0.0034  +0.0033  +0.0199  ✅
  attention > averaging                   +0.0029  -0.0063  -0.0090  +0.0131  ⚠️
  attention > stacking                    +0.0237  +0.0018  +0.0048  +0.0300  ⚠️
  attention > gating                      +0.0012  -0.0069  -0.0095  +0.0116  ⚠️
  attention > moe                         +0.0030  -0.0035  -0.0062  +0.0315  ✅

--- C2 test ---
  Comparison                        

## Cell 5 — Paired bootstrap function

In [5]:
def paired_bootstrap(y_true, probs_a, probs_b, metric_fn,
                     B=B_RESAMPLES, rng=None):
    """
    One-tailed paired bootstrap: H1: metric(A) > metric(B).
    Efron-Tibshirani (1993) shift trick centres the null distribution.
    """
    if rng is None:
        rng = np.random.default_rng(RNG_SEED)
    y_true, probs_a, probs_b = (np.asarray(x) for x in (y_true, probs_a, probs_b))
    n = len(y_true)

    metric_a  = float(metric_fn(y_true, probs_a))
    metric_b  = float(metric_fn(y_true, probs_b))
    delta_obs = metric_a - metric_b

    boot_deltas = np.empty(B, dtype=np.float64)
    for i in range(B):
        idx = rng.integers(0, n, size=n)
        boot_deltas[i] = (metric_fn(y_true[idx], probs_a[idx])
                          - metric_fn(y_true[idx], probs_b[idx]))

    p_value = float(np.mean(boot_deltas <= 0.0))
    ci_low  = float(np.percentile(boot_deltas, 2.5))
    ci_high = float(np.percentile(boot_deltas, 97.5))

    sig = ('***' if p_value < 0.001 else
           '**'  if p_value < 0.01  else
           '*'   if p_value < 0.05  else 'ns')

    return dict(delta_obs=round(delta_obs,6), p_value=round(p_value,6),
                ci_low=round(ci_low,6), ci_high=round(ci_high,6),
                metric_a=round(metric_a,6), metric_b=round(metric_b,6), sig=sig)

METRIC_FNS = {
    'f1':          lambda yt,yp: float(f1_score(yt,(yp>=0.5).astype(int),zero_division=0)),
    'auroc':       lambda yt,yp: float(roc_auc_score(yt, yp)),
    'pr_auc':      lambda yt,yp: float(average_precision_score(yt, yp)),
    'tpr_at_1fpr': lambda yt,yp: _tpr_at_fpr(yt, yp, 0.01),
}
METRIC_NAMES = {'f1':'F1','auroc':'AUROC','pr_auc':'PR-AUC','tpr_at_1fpr':'TPR@1%FPR'}
print('Bootstrap function defined.  B={:,}  seed={}'.format(B_RESAMPLES, BOOTSTRAP_SEED))


Bootstrap function defined.  B=10,000  seed=42


## Cell 6 — Run bootstrap (seed=42 partition)

Comparisons run:
1. **Gating vs. all other fusion strategies** — balanced + C2 test
2. **Gating vs. XGBoost / CodeBERT / GATv2** — balanced + C2 test
3. **Attention vs. all other fusion strategies** — C2 only (for TPR@1%FPR claim)

Runtime: ~15–20 min on CPU.


In [6]:
rng = np.random.default_rng(RNG_SEED)
BS_RESULTS = {}
t0 = time.time()

print(f'Running paired bootstrap  B={B_RESAMPLES:,}  seed={BOOTSTRAP_SEED}  α={ALPHA}')
print('~15-20 min on CPU\n')

RUNS = [
    # (ref_key, splits_to_run, opponents_dict)
    (PRIMARY_REF,
     ['test', 'test_c2'],
     {**{s: PROBS[s] for s in STRATEGIES if s != PRIMARY_REF}, **BRANCH}),
    (SECONDARY_REF,
     ['test_c2'],
     {s: PROBS[s] for s in STRATEGIES if s != SECONDARY_REF}),
]

for ref_key, splits, opponents in RUNS:
    for split in splits:
        y_true  = LABELS[split]
        p_ref   = PROBS[ref_key][split]
        slabel  = 'Balanced' if split == 'test' else 'C2'
        key     = f'{ref_key}_vs_others_{split}'
        BS_RESULTS[key] = {}

        for opp_name, opp_probs_dict in opponents.items():
            p_opp = opp_probs_dict[split]
            results = {}
            for metric, fn in METRIC_FNS.items():
                results[metric] = paired_bootstrap(y_true, p_ref, p_opp, fn, rng=rng)
            BS_RESULTS[key][opp_name] = results
            elapsed = time.time() - t0
            f1r = results['f1']
            print(f'  [{slabel}] {ref_key} vs {opp_name:<22} '
                  f'{elapsed:>6.0f}s  F1: ref={f1r["metric_a"]:.4f} other={f1r["metric_b"]:.4f}'
                  f'  p={f1r["p_value"]:.4f}{f1r["sig"]}')

print(f'\nTotal: {(time.time()-t0)/60:.1f} min')
with open(BOOTSTRAP_PATH, 'w') as f:
    json.dump(BS_RESULTS, f, indent=2)
print(f'Saved: {BOOTSTRAP_PATH}')


Running paired bootstrap  B=10,000  seed=42  α=0.05
~15-20 min on CPU

  [Balanced] gating vs averaging                 128s  F1: ref=0.9724 other=0.9707  p=0.2343ns
  [Balanced] gating vs stacking                  255s  F1: ref=0.9724 other=0.9499  p=0.0000***
  [Balanced] gating vs attention                 382s  F1: ref=0.9724 other=0.9736  p=1.0000ns
  [Balanced] gating vs moe                       507s  F1: ref=0.9724 other=0.9706  p=0.2837ns
  [Balanced] gating vs XGBoost                   640s  F1: ref=0.9724 other=0.9724  p=1.0000ns
  [Balanced] gating vs CodeBERT                  771s  F1: ref=0.9724 other=0.7671  p=0.0000***
  [Balanced] gating vs GATv2                     904s  F1: ref=0.9724 other=0.9238  p=0.0000***
  [C2] gating vs averaging                1018s  F1: ref=0.9175 other=0.9373  p=0.9736ns
  [C2] gating vs stacking                 1132s  F1: ref=0.9175 other=0.7320  p=0.0000***
  [C2] gating vs attention                1248s  F1: ref=0.9175 other=0.9211  p=1.

## Cell 7 — Paper-ready Table 4

In [7]:
def print_bs_table(key, ref_label, split_label):
    results = BS_RESULTS.get(key, {})
    if not results:
        print(f'No results for {key}'); return
    print(f'\n{"="*82}')
    print(f'  {ref_label}  vs.  others  |  {split_label}  (seed=42 partition)')
    print(f'  Paired bootstrap  B={B_RESAMPLES:,}  one-tailed H1: ref > other  α={ALPHA}')
    print(f'  * p<0.05   ** p<0.01   *** p<0.001   ns = not significant')
    print(f'{"="*82}')
    hdr = (f'{"Opponent":<26}  {"Metric":<14}'
           f'  {"Ref":>8}  {"Other":>8}  {"Δ":>8}'
           f'  {"p":>7}  {"95% CI":>24}  Sig')
    print(hdr)
    print('-'*len(hdr))
    for opp, mres in results.items():
        opp_label = STRAT_LABELS.get(opp, opp)
        first = True
        for metric in ['f1','auroc','pr_auc','tpr_at_1fpr']:
            r  = mres[metric]
            ci = f'[{r["ci_low"]:+.4f}, {r["ci_high"]:+.4f}]'
            ol = opp_label if first else ''
            print(f'  {ol:<24}  {METRIC_NAMES[metric]:<14}'
                  f'  {r["metric_a"]:>8.4f}  {r["metric_b"]:>8.4f}'
                  f'  {r["delta_obs"]:>+8.4f}  {r["p_value"]:>7.4f}'
                  f'  {ci:>24}  {r["sig"]:>3}')
            first = False
        print()

print_bs_table(f'{PRIMARY_REF}_vs_others_test',
               STRAT_LABELS[PRIMARY_REF], 'Balanced test')
print_bs_table(f'{PRIMARY_REF}_vs_others_test_c2',
               STRAT_LABELS[PRIMARY_REF], 'C2 test (\u226410% mal)')
print_bs_table(f'{SECONDARY_REF}_vs_others_test_c2',
               STRAT_LABELS[SECONDARY_REF], 'C2 test (\u226410% mal)')

# LaTeX summary
print('\n' + '='*72)
print('LaTeX p-value summary — Gating vs. others on C2 (for Table 4 notes)')
print(f'{"Opponent":<26}  F1          AUROC       PR-AUC      TPR@1%FPR')
print('-'*72)
for opp, mres in BS_RESULTS.get(f'{PRIMARY_REF}_vs_others_test_c2', {}).items():
    opp_l = STRAT_LABELS.get(opp, opp)
    parts = []
    for m in ['f1','auroc','pr_auc','tpr_at_1fpr']:
        r = mres[m]
        parts.append(f'p={r["p_value"]:.4f}{r["sig"]}')
    print(f'  {opp_l:<24}  {"  ".join(parts)}')



  Gating (softmax weights)  vs.  others  |  Balanced test  (seed=42 partition)
  Paired bootstrap  B=10,000  one-tailed H1: ref > other  α=0.05
  * p<0.05   ** p<0.01   *** p<0.001   ns = not significant
Opponent                    Metric               Ref     Other         Δ        p                    95% CI  Sig
----------------------------------------------------------------------------------------------------------------
  Averaging (geom. mean)    F1                0.9724    0.9707   +0.0017   0.2343        [-0.0030, +0.0063]   ns
                            AUROC             0.9962    0.9956   +0.0006   0.1354        [-0.0005, +0.0017]   ns
                            PR-AUC            0.9963    0.9958   +0.0005   0.1753        [-0.0006, +0.0016]   ns
                            TPR@1%FPR         0.9439    0.9424   +0.0015   0.5028        [-0.0355, +0.0287]   ns

  Stacking (LR on probs)    F1                0.9724    0.9499   +0.0225   0.0000        [+0.0152, +0.0299]  ***
   

## Cell 8 — Sanity checks

In [8]:
checks = []
def chk(ok, name, detail=''):
    checks.append((bool(ok), name, str(detail)))

# 1. Output file written
chk(BOOTSTRAP_PATH.exists(), 'bootstrap_results.json written', str(BOOTSTRAP_PATH))

# 2. All expected keys present
for k in [f'{PRIMARY_REF}_vs_others_test', f'{PRIMARY_REF}_vs_others_test_c2',
          f'{SECONDARY_REF}_vs_others_test_c2']:
    chk(k in BS_RESULTS, f'Key present: {k}')

# 3. Reference metrics are sane (F1 > 0.9 on balanced, > 0.85 on C2)
for split, thresh in [('test', 0.90), ('test_c2', 0.85)]:
    key = f'{PRIMARY_REF}_vs_others_{split}'
    if key in BS_RESULTS:
        first_opp = list(BS_RESULTS[key].keys())[0]
        ref_f1 = BS_RESULTS[key][first_opp]['f1']['metric_a']
        chk(ref_f1 >= thresh,
            f'Gating ref F1 on {split} >= {thresh} (probs are correct)',
            f'ref_f1={ref_f1:.4f}')

# 4. Branch comparisons included
branch_found = [b for b in BS_RESULTS.get(f'{PRIMARY_REF}_vs_others_test', {})
                if b in ('XGBoost', 'CodeBERT', 'GATv2')]
chk(len(branch_found) == 3, 'All 3 branch baselines compared', str(branch_found))

# 5. At least one comparison is significant (expected given the metric gaps)
key_c2 = f'{PRIMARY_REF}_vs_others_test_c2'
n_sig = sum(r['p_value'] < ALPHA
            for opp, mres in BS_RESULTS.get(key_c2, {}).items()
            for r in mres.values())
chk(n_sig > 0, f'At least 1 significant result on C2 (p<{ALPHA})',
    f'{n_sig} significant comparisons')

# 6. Gating vs XGBoost: gating should be significantly better on C2 F1
xgb_c2 = BS_RESULTS.get(key_c2, {}).get('XGBoost', {}).get('f1', {})
if xgb_c2:
    chk(xgb_c2.get('p_value', 1.0) < ALPHA,
        'Gating C2 F1 significantly > XGBoost C2 F1',
        f'p={xgb_c2.get("p_value","?"):.4f}  '
        f'Δ={xgb_c2.get("delta_obs","?"):.4f}')

passed = sum(1 for ok,_,_ in checks if ok)
failed = sum(1 for ok,_,_ in checks if not ok)
print('\n=== BOOTSTRAP SANITY CHECKS ===')
for ok, name, detail in checks:
    print(f'  {"\u2705" if ok else "\u274c"}  {name}')
    if detail: print(f'       {detail}')
print(f'\n{passed} passed, {failed} failed.')
if failed == 0:
    print('Upload /kaggle/working/bootstrap/ as a Kaggle dataset.')
    print('Use bootstrap_results.json for Table 4 in the paper.')
else:
    print('Fix failures before using results in the paper.')



=== BOOTSTRAP SANITY CHECKS ===
  ✅  bootstrap_results.json written
       /kaggle/working/bootstrap/bootstrap_results.json
  ✅  Key present: gating_vs_others_test
  ✅  Key present: gating_vs_others_test_c2
  ✅  Key present: attention_vs_others_test_c2
  ✅  Gating ref F1 on test >= 0.9 (probs are correct)
       ref_f1=0.9724
  ✅  Gating ref F1 on test_c2 >= 0.85 (probs are correct)
       ref_f1=0.9175
  ✅  All 3 branch baselines compared
       ['XGBoost', 'CodeBERT', 'GATv2']
  ✅  At least 1 significant result on C2 (p<0.05)
       13 significant comparisons
  ❌  Gating C2 F1 significantly > XGBoost C2 F1
       p=1.0000  Δ=0.0000

8 passed, 1 failed.
Fix failures before using results in the paper.
